# AF2 Curriculum-SFS — seed-42 kill screen

Melatih **satu kandidat AF2CURR1** dari checkpoint AF2 seed-42. SFS diramp bertahap dan auxiliary pure-gate hanya aktif ketika gradiennya searah dengan detection loss. Pembanding adalah AF2CTRL dengan parent SHA dan jadwal yang sama. Tidak membuka test dan tidak menjalankan seed tambahan.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='codex/af2-curriculum-sfs'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result=subprocess.run(clone,cwd='/content')
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    time.sleep(2)
else: raise RuntimeError('git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True,cwd='/content')
SRC=str(REPO/'src')
if SRC not in sys.path: sys.path.insert(0,SRC)
importlib.invalidate_caches()
for key in list(sys.modules):
    if key=='coffee_detector' or key.startswith('coffee_detector.'): sys.modules.pop(key,None)
os.chdir(REPO)
import coffee_detector, torch
assert Path(coffee_detector.__file__).resolve().is_relative_to(REPO.resolve()),coffee_detector.__file__
assert torch.cuda.is_available(),'Aktifkan GPU Colab.'
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
required=(
 'bundles/faruq-development-v3-grouped.tar',
 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt',
 'experiments/faruq-v3-af2-complement-v1/val_reports/AF2CTRL_seed42_result.json',
)
PROJECT=resolve_drive_project_root(required_relative_paths=required)
ARCHIVE=require_project_artifact(PROJECT,required[0])
AF2=require_project_artifact(PROJECT,required[1])
CONTROL=require_project_artifact(PROJECT,required[2])
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as stream: stream.extractall('/content',filter='data')
GROUPED=DATA/'faruq_grouped_summary.json'
assert GROUPED.is_file() and not (DATA/'test').exists()
OUTPUT=PROJECT/'experiments/faruq-v3-af2-curriculum-sfs-v1'
OUTPUT.mkdir(parents=True,exist_ok=True)
print('DATA:',DATA)
print('AF2:',AF2)
print('CONTROL:',CONTROL)
print('OUTPUT:',OUTPUT)

In [ ]:
from coffee_detector.af2_curriculum_sfs import run_af2_curriculum_sfs_static_audit
STATIC=OUTPUT/'static_audit.json'
audit=run_af2_curriculum_sfs_static_audit(AF2,STATIC,device='0')
print('PARAMETERS:',audit['parameters'])
print('INITIAL DIFF:',audit['initial_output_max_abs_diff'])
print('SCHEDULE:',audit['schedule'])
print('GATES:',audit['gates'])
print('DECISION:',audit['decision'])
assert audit['decision']=='PASS','STOP: static audit gagal; jangan training.'

In [ ]:
ARM='AF2CURR1'; LOG=OUTPUT/f'{ARM}_seed42_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_curriculum_sfs',
 '--data-root',str(DATA),'--grouped-summary',str(GROUPED),
 '--af2-checkpoint',str(AF2),'--af2ctrl-result',str(CONTROL),
 '--static-audit',str(STATIC),'--output-root',str(OUTPUT),
 '--seed','42','--device','0','--authorize-training']
print('START/RESUME:',ARM,'| log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream:
    process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
last_status=None
while process.poll() is None:
    csv_path=OUTPUT/ARM/f'{ARM}_seed42/results.csv'
    epochs=max(0,sum(1 for _ in csv_path.open())-1) if csv_path.is_file() else 0
    status=f'{ARM}: {epochs}/30 epoch tercatat'
    if status!=last_status: print(status,flush=True); last_status=status
    time.sleep(60)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-150:]))
    raise RuntimeError(f'{ARM} gagal: {process.returncode}')
SUMMARY=OUTPUT/'val_reports/af2_curriculum_sfs_seed42_decision.json'
summary=json.loads(SUMMARY.read_text())
print('VALUES:',summary['values'])
print('DELTAS:',summary['deltas'])
print('CRITERIA:',summary['criteria'])
print('DECISION:',summary['decision'])
print('NEXT:',summary['next'])
print('TEST:',summary['test_opened'])
print('Kirim values, deltas, criteria, decision, dan next. Jangan membuka test.')